### A Chat Agent That Knows Your Customers

Welcome to this tutorial on building a chat agent that can search your customer data with the M|BOX Python SDK. Support staff ask about customers the way they hear them on the phone: names are misspelled, companies are shortened, and two customers can share the same name. A chat agent is only useful if it finds the right record anyway, and if it remembers who the conversation is about.

We will give an agent two tools: a fuzzy `search_customers` tool that performs identity resolution with an M|BOX index, and a `get_customer` tool that returns the full account record. Then we will run a short support conversation with the OpenAI Agents SDK, using a session so the agent remembers every earlier message.

This tutorial calls the OpenAI API, so it needs an `OPENAI_API_KEY` in a `.env` file in the current folder or one of its parent folders. Our customer data is stored in `dataset/customers.csv`. Let's load it.

In [1]:
import time

import pandas as pd
from mbox import TableIndexer

customers = pd.read_csv("datasets/customers-5k.csv")

print(f"{len(customers):,} customers")
customers.head()

5,000 customers


,customer_id,first_name,last_name,company,city,email,phone,plan,open_tickets,customer_since
0,C20001,Melissa,Lang,Lumio Solutions,Dresden,melissa.lang@lumio.com,+49 175 4715621,Basic,0,2025
1,C20002,Raymond,Brooks,Clearwater Manufacturing,Munich,raymond.brooks@clearwater.com,+49 173 3794988,Enterprise,0,2020
2,C20003,Michael,Albrecht,Falcon Systems,Hamburg,michael.albrecht@falcon.com,+49 177 6096798,Pro,3,2024
3,C20004,Michael,Fischer,Silverline Manufacturing,Bremen,michael.fischer@silverline.com,+49 161 5529852,Pro,0,2019
4,C20005,Margaret,Weber,Oakridge Logistics,Konstanz,margaret.weber@oakridge.com,+49 179 5845129,Basic,1,2015


The table holds 5,000 customers. Each row has the details people use to identify a customer, `first_name`, `last_name`, `company` and `city`, and the account details support staff need once the customer is found: `email`, `phone`, `plan`, `open_tickets` and `customer_since`.

Like any real customer base, it contains a trap: two different customers named `Robert Hayes`, one in `Hamburg` and one in `Munich`. We will meet them during the conversation.

### Building the Search Index

Only the four identifying columns need to be searchable. Let's build an M|BOX index on them and time it.

In [2]:
SEARCH_FIELDS = ["first_name", "last_name", "company", "city"]

start = time.perf_counter()
index = TableIndexer.create_index(customers, index_columns=SEARCH_FIELDS, tmp_dir="tmp_index")

print(f"Indexed {len(customers):,} customers in {time.perf_counter() - start:.2f} seconds")

Indexed 5,000 customers in 0.13 seconds


M|BOX indexed all 5,000 customers in 0.11 seconds. The account columns were not included in `index_columns`, so they are stored but not searched, and every match still returns them.

### The Search Tool

The first tool is the identity resolution step. `search_customers()` takes whatever the caller said, runs `index.match()`, and returns up to 3 candidates with an overall `score`, the score of each field, and the time the search took. Unknown fields are passed as `""`, and M|BOX leaves them out of the overall score. Let's define it.

In [3]:
def search_customers(first_name: str, last_name: str, company: str, city: str) -> dict:
    """Search the customer database. Tolerates typos, partial company names and missing fields.

    Args:
        first_name: The customer's first name, or "" if unknown.
        last_name: The customer's last name, or "" if unknown.
        company: The customer's company, or "" if unknown.
        city: The customer's city, or "" if unknown.
    """
    start = time.perf_counter()
    matches = index.match(
        first_name=first_name,
        last_name=last_name,
        company=company,
        city=city,
        max_results=3,
        include_field_scores=True
    )
    search_ms = (time.perf_counter() - start) * 1000

    candidates = [
        {
            "customer_id": row["customer_id_candidate"],
            "name": f"{row['first_name_candidate']} {row['last_name_candidate']}",
            "company": row["company_candidate"],
            "city": row["city_candidate"],
            "score": int(row["overall_score"]),
            "field_scores": {field: int(row[f"{field}_score"]) for field in SEARCH_FIELDS},
        }
        for _, row in matches.iterrows()
        if row["index_row"] >= 0
    ]
    return {"candidates": candidates, "search_ms": round(search_ms, 2)}

The type hints and the docstring describe the tool to the model, so each argument explains what to pass when a detail is unknown. Returning several candidates matters in a chat: when two customers match equally well, the agent has to see both to ask the right follow-up question.

### Testing the Search Tool

Before adding an LLM, let's call the tool ourselves the way a support agent might type it: a misspelled first name and only the first word of the company.

In [4]:
import json

print(json.dumps(search_customers("Kathrine", "Voss", "Kestrel", ""), indent=2))

{
  "candidates": [
    {
      "customer_id": "C20813",
      "name": "Katherine Voss",
      "company": "Kestrel Manufacturing",
      "city": "Stuttgart",
      "score": 89,
      "field_scores": {
        "first_name": 78,
        "last_name": 100,
        "company": 90,
        "city": 0
      }
    }
  ],
  "search_ms": 2.78
}


M|BOX found `Katherine Voss` at `C20813` with a score of 89 in 2.78 milliseconds. The field scores explain the result:
- `first_name` scores 78, because `Kathrine` is missing one letter.
- `last_name` scores 100, an exact match.
- `company` scores 90, since `Kestrel` is only part of `Kestrel Manufacturing`.
- `city` scores 0 because we left it empty, and it does not count toward the overall score.

### The Account Details Tool

Once a customer is identified, the agent needs their account details. This is an exact lookup by `customer_id`, so plain pandas is the right tool here, not a fuzzy search. Let's define `get_customer()` and try it with the id we just found.

In [6]:
def get_customer(customer_id: str) -> dict:
    """Get the full account record of one customer: contact details, plan, open tickets.

    Args:
        customer_id: The exact customer_id returned by search_customers, for example C20813.
    """
    record = customers[customers["customer_id"] == customer_id]
    if record.empty:
        return {"error": f"No customer with id {customer_id}"}
    return record.iloc[0].to_dict()


get_customer("C20813")

{'customer_id': 'C20813',
 'first_name': 'Katherine',
 'last_name': 'Voss',
 'company': 'Kestrel Manufacturing',
 'city': 'Stuttgart',
 'email': 'katherine.voss@kestrel.com',
 'phone': '+49 171 5387923',
 'plan': 'Enterprise',
 'open_tickets': 2,
 'customer_since': 2018}

`get_customer()` returned the full record for `C20813`: an `Enterprise` plan, 2 open tickets, and a customer since 2018. Splitting the work into two tools keeps each one simple. M|BOX answers "who is this?", and the second tool answers "what do we know about them?" for exactly one id.

### Creating the Chat Agent

Now let's create the agent with both tools. The instructions define when a customer counts as identified: a best score of 80 or more, with no other candidate within 10 points. They also tell the agent to never guess an id, to ask for one more detail when candidates are close, and to keep its answers short like a colleague in a chat.

In [7]:
from agents import Agent, ModelSettings, Runner, SQLiteSession, function_tool
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

INSTRUCTIONS = """You are a support assistant in a live chat with our support staff.
To identify a customer, call search_customers with only the details you were given, and "" for the rest.
Call search_customers again whenever you learn a new detail about the customer.
A customer is identified when the best candidate scores 80 or more and no other candidate is within 10 points.
If several candidates are close, list them briefly and ask for one detail that tells them apart.
Use get_customer only for an identified customer_id, and never guess an id.
If there are no candidates, say that the person is not in our customer database.
Keep answers short, like a colleague in a chat."""

agent = Agent(
    name="Support chat",
    instructions=INSTRUCTIONS,
    tools=[function_tool(search_customers), function_tool(get_customer)],
    model="gpt-4.1-mini",
    model_settings=ModelSettings(temperature=0)
)

Notice that the rules for accepting a match are written in plain language, while the matching itself stays in M|BOX. The model decides what to search for and how to reply. It never decides on its own which record is correct.

### A Chat with Memory

A chat is a sequence of messages, and each new message builds on the earlier ones. The Agents SDK handles this with a session. A `SQLiteSession` stores every message, tool call and tool result, and passes the history back to the model on each turn. Our `chat()` helper runs one turn in the session and prints the tool calls, the search time, and the agent's reply. Let's start the conversation.

In [8]:
session = SQLiteSession("support_chat")


async def chat(message):
    start = time.perf_counter()
    result = await Runner.run(agent, message, session=session)
    print(f"Staff: {message}\n")
    for item in result.new_items:
        if item.type == "tool_call_item":
            print(f"  [tool] {item.raw_item.name}({item.raw_item.arguments})")
        elif item.type == "tool_call_output_item" and "search_ms" in item.output:
            print(f"  [tool] {len(item.output['candidates'])} candidate(s) in {item.output['search_ms']} ms")
    print(f"\nAgent ({time.perf_counter() - start:.1f} s): {result.final_output}")


await chat("Hi, I have Kathrine Voss from Kestrel on the phone. Can you pull her up?")

Staff: Hi, I have Kathrine Voss from Kestrel on the phone. Can you pull her up?

  [tool] search_customers({"first_name":"Kathrine","last_name":"Voss","company":"Kestrel","city":""})
  [tool] 1 candidate(s) in 3.97 ms
  [tool] get_customer({"customer_id":"C20813"})

Agent (6.0 s): I found Katherine Voss from Kestrel Manufacturing in Stuttgart. She has an Enterprise plan and currently 2 open tickets. How can I assist you with her account?


In a single turn, the agent chained both tools. It called `search_customers` with exactly the details from the message, M|BOX found one candidate in 3.72 milliseconds, and because the match was clear, the agent called `get_customer` with `C20813` and replied with her plan and open tickets.

Notice the timing: the search took a few milliseconds, while the whole turn took 4.5 seconds. Nearly all of that time is spent waiting for the language model.

### Follow-up Questions

Now let's ask a follow-up question that does not repeat her name.

In [8]:
await chat("What is her email, and how long has she been a customer?")

Staff: What is her email, and how long has she been a customer?


Agent (1.2 s): Her email is katherine.voss@kestrel.com, and she has been a customer since 2018.


The agent answered without calling any tool at all. Her email and the year she became a customer were already in the `get_customer` result from the previous turn, and the session passed that result back to the model. This is what makes it a chat instead of a set of unrelated questions: `she` is understood, and the answer still comes from the record, not from the model's imagination.

### Two Customers with the Same Name

Let's move on to the next caller.

In [9]:
await chat("Thanks. Next caller is Robert Hayes, he wants to know his plan.")

Staff: Thanks. Next caller is Robert Hayes, he wants to know his plan.

  [tool] search_customers({"first_name":"Robert","last_name":"Hayes","company":"","city":""})
  [tool] 3 candidate(s) in 3.21 ms

Agent (2.9 s): There are two Robert Hayes in the system: one from Voltra International in Hamburg, and one from Lumio Solutions in Munich. Which one is the caller?


M|BOX returned 3 candidates, and the top two are both named `Robert Hayes`, each with a perfect score. Following its instructions, the agent did not pick one. It named both, one in `Hamburg` and one in `Munich`, and asked which one is calling. Answering with the wrong customer's plan would be worse than asking.

Let's give the agent the detail it asked for.

In [ ]:
await chat("He says he is based in Munich.")

Staff: He says he is based in Munich.

  [tool] get_customer({"customer_id":"C23979"})

Agent (2.5 s): Robert Hayes from Lumio Solutions in Munich is on the Basic plan.


The agent went straight to `get_customer` with `C23979`, the `Robert Hayes` from `Lumio Solutions` in `Munich`, and replied that he is on the `Basic` plan. It did not need to search again, because both candidates and their cities were already in the session from the previous turn. The id still came from M|BOX, never from a guess.

### A Caller Who Is Not a Customer

Finally, let's ask about someone who is not in our customer data.

In [11]:
await chat("Last one: Laura Chen from Brightwave says she is a customer. Can you find her?")

Staff: Last one: Laura Chen from Brightwave says she is a customer. Can you find her?

  [tool] search_customers({"first_name":"Laura","last_name":"Chen","company":"Brightwave","city":""})
  [tool] 0 candidate(s) in 2.88 ms

Agent (2.3 s): Laura Chen from Brightwave is not in our customer database.


M|BOX returned 0 candidates in 2.88 milliseconds, and the agent said clearly that `Laura Chen` from `Brightwave` is not in the customer database. M|BOX does not force a match when nothing is close enough, so the agent has no record to attach to a stranger.

### What the Session Remembers

Let's look at what the session stored during the conversation.

In [12]:
history = await session.get_items()

print(f"{len(history)} items stored in the session")
print(sorted({item.get("type", "message") for item in history}))

20 items stored in the session
['function_call', 'function_call_output', 'message']


The session holds 20 items from five turns. They are of three types: the chat `message` items, every `function_call` the agent made, and every `function_call_output` it received. Because tool results are part of the history, the agent could answer follow-up questions from records it had already retrieved. By default `SQLiteSession` keeps this in memory, and passing a file path as `db_path` stores conversations on disk instead.

### Summary

In this tutorial, we built a support chat agent that searches customer data with M|BOX:
- `TableIndexer.create_index()` indexed 5,000 customers on their identifying fields in a fraction of a second, and each search took a few milliseconds.
- A `search_customers` tool performs identity resolution with `index.match()`, tolerating typos and partial company names, and returns several candidates with field scores.
- A separate `get_customer` tool returns the account record for one exact `customer_id`, so the agent only fetches details for a customer M|BOX has identified.
- A `SQLiteSession` stores messages, tool calls and tool results, so the agent understands follow-up questions and can reuse records it already retrieved.
- When two customers match equally well the agent asks for one more detail, and when nothing matches it says so instead of guessing.